# 03 — Feature Engineering

**Primary author:** Victoria

**Builds on:**
- *02_embedding_generation.ipynb* (Victoria — embedding files and index contract)
- Hans's *Hans_Supervised_Learning.ipynb* and *Hans_Supervised_Learning_Models.ipynb*
  (cosine similarity computation pattern, WordNet relationship features, surface
  features — expanding from 10 to 46)

**Prompt engineering:** Victoria  
**AI assistance:** Claude Code (Anthropic)  
**Environment:** Local (CPU only)

Compute all 46 features for each (clue, definition, answer) row. Loads
embeddings from Step 2, computes 15 context-free + 6 context-informed cosine
similarities, 21 WordNet relationship features, and 4 surface features.
Outputs `data/features_all.parquet`.

Implements PLAN.md Step 3.

**Inputs:**
- `data/clues_filtered.csv` (Step 1)
- `data/embeddings/` — 3 `.npy` arrays + 3 index CSVs (Step 2)
- `data/embeddings/clue_context_phrases.csv` (Step 2 — provides `definition_wn`
  and `answer_wn` lookup keys)
- WordNet (via NLTK)

**Output:**
- `data/features_all.parquet` — one row per (clue, definition, answer) with
  all 46 features plus metadata columns

## Imports

In [ ]:
import warnings

import numpy as np
import pandas as pd
from pathlib import Path

import nltk
from nltk.corpus import wordnet as wn
from sklearn.metrics.pairwise import cosine_similarity

from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)

## Environment and Paths

In [ ]:
# --- Environment Auto-Detection ---
# Same pattern as 02_embedding_generation.ipynb: detect Colab, Great Lakes,
# or local and set paths accordingly.
try:
    IS_COLAB = 'google.colab' in str(get_ipython())
except NameError:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/SIADS 692 Milestone II/'
                        'Milestone II - NLP Cryptic Crossword Clues/'
                        'clue_misdirection')
else:
    # Local or Great Lakes: notebook is in clue_misdirection/notebooks/,
    # so parent is the clue_misdirection project root.
    try:
        PROJECT_ROOT = Path(__file__).resolve().parent.parent
    except NameError:
        PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / 'data'
EMBEDDINGS_DIR = DATA_DIR / 'embeddings'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Download WordNet data — needed for the 21 relationship features computed
# later in this notebook.
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print(f'Environment: {"Google Colab" if IS_COLAB else "Local / Great Lakes"}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Data directory: {DATA_DIR}')
print(f'Embeddings directory: {EMBEDDINGS_DIR}')

## Load Data

We load two CSV files:

1. **`clues_filtered.csv`** (Step 1) — 241,397 rows with all metadata columns
   (`clue_id`, `clue`, `surface`, `definition`, `answer`, `def_answer_pair_id`,
   etc.).

2. **`clue_context_phrases.csv`** (Step 2) — 240,211 rows that survived the
   Step 2 cleanup (dropping 1,186 rows for duplicate definitions in the surface
   text and fully unresolvable words). This file provides the `definition_wn`
   and `answer_wn` columns — the WordNet-ready lookup keys needed to find the
   correct row in the embedding index files.

We inner-merge on `(clue_id, definition)` to produce a 240,211-row working
set that has both the original metadata and the embedding lookup keys.
Merging on `clue_id` alone would be wrong: double-definition clues have
multiple rows per `clue_id` (one per valid definition), so a `clue_id`-only
join would produce a many-to-many cross product. Adding `definition` as a
second key disambiguates these rows.

We also record each row's position in `clue_context_phrases.csv` as
`cc_row_position`. Since `clue_context_phrases.csv` and
`clue_context_embeddings.npy` are in identical row order (verified in
Step 2), this gives us a direct index into the clue-context embedding
array — no need to map through `clue_context_index.csv`, which only has
`clue_id` and would suffer the same double-definition ambiguity.

In [ ]:
# --- Load clues_filtered.csv (Step 1 output) ---
clues_path = DATA_DIR / 'clues_filtered.csv'
assert clues_path.exists(), (
    f'Missing input file: {clues_path}\n'
    f'Run 01_data_cleaning.ipynb first to produce this file.'
)
clues_df = pd.read_csv(clues_path)
print(f'clues_filtered.csv: {len(clues_df):,} rows')

# --- Load clue_context_phrases.csv (Step 2 intermediate) ---
# This file provides definition_wn and answer_wn — the WordNet-ready lookup
# keys that map to the embedding index files. It also identifies which rows
# survived Step 2's cleanup (240,211 of the original 241,397).
# CRITICAL: keep_default_na=False prevents pandas from interpreting the word
# "nan" (grandmother) as NaN — see DATA.md.
cc_phrases_path = EMBEDDINGS_DIR / 'clue_context_phrases.csv'
assert cc_phrases_path.exists(), (
    f'Missing input file: {cc_phrases_path}\n'
    f'Run 02_embedding_generation.ipynb first to produce this file.'
)
cc_phrases = pd.read_csv(cc_phrases_path, keep_default_na=False)
print(f'clue_context_phrases.csv: {len(cc_phrases):,} rows')

# --- Record each row's position in cc_phrases / clue_context_embeddings ---
# clue_context_phrases.csv and clue_context_embeddings.npy are in identical
# row order (verified in Step 2). By recording the row position here, we get
# a direct index into the clue-context embedding array after the merge —
# avoiding the ambiguity of mapping through clue_context_index.csv, which
# only has clue_id and can't disambiguate double-definition clues.
cc_phrases['cc_row_position'] = np.arange(len(cc_phrases))

# --- Merge to get definition_wn and answer_wn onto the clue rows ---
# Inner merge restricts to the 240,211 rows that have embeddings.
# We merge on (clue_id, definition) — NOT clue_id alone — because
# double-definition clues have multiple rows per clue_id. A clue_id-only
# merge would produce a many-to-many cross product for those clues,
# inflating the row count. The 'definition' column appears in both files
# (original-case definition text) and disambiguates which definition each
# row corresponds to.
df = clues_df.merge(
    cc_phrases[['clue_id', 'definition', 'definition_wn', 'answer_wn',
                'def_num_usable_synsets', 'ans_num_usable_synsets',
                'cc_row_position']],
    on=['clue_id', 'definition'],
    how='inner'
)

# Verify the merge produced exactly the expected number of rows — no
# inflation from many-to-many joins and no unexpected drops.
assert len(df) == len(cc_phrases), (
    f'Merge produced {len(df):,} rows, expected {len(cc_phrases):,}. '
    f'This likely means a double-definition clue was not disambiguated '
    f'correctly by the (clue_id, definition) key.')

print(f'\nWorking set after merge: {len(df):,} rows')
print(f'  (dropped {len(clues_df) - len(df):,} rows without embeddings)')
print(f'  Unique (definition, answer) pairs: '
      f'{df["def_answer_pair_id"].nunique():,}')
print(f'\nColumns: {list(df.columns)}')
df.head(3)

## Load Embeddings

Load all 6 embedding files from Step 2 (3 `.npy` arrays + 3 index CSVs).
The index CSVs map row positions in the `.npy` arrays to word strings
(`definition_wn` / `answer_wn`) or `clue_id`.

In [ ]:
# Load embedding arrays and index files.
# CRITICAL: keep_default_na=False on all index CSVs — the word "nan"
# (grandmother) is a valid crossword definition/answer.

definition_embeddings = np.load(EMBEDDINGS_DIR / 'definition_embeddings.npy')
definition_index = pd.read_csv(
    EMBEDDINGS_DIR / 'definition_index.csv', index_col=0,
    keep_default_na=False)

answer_embeddings = np.load(EMBEDDINGS_DIR / 'answer_embeddings.npy')
answer_index = pd.read_csv(
    EMBEDDINGS_DIR / 'answer_index.csv', index_col=0,
    keep_default_na=False)

clue_context_embeddings = np.load(
    EMBEDDINGS_DIR / 'clue_context_embeddings.npy')
clue_context_index = pd.read_csv(
    EMBEDDINGS_DIR / 'clue_context_index.csv', index_col=0,
    keep_default_na=False)

# --- Print shapes and sizes ---
EMBED_DIM = 1024
print(f'{"File":<35s} {"Shape":<25s} {"Memory":>8s}')
print(f'{"-"*35} {"-"*25} {"-"*8}')
for name, arr in [
    ('definition_embeddings.npy', definition_embeddings),
    ('answer_embeddings.npy', answer_embeddings),
    ('clue_context_embeddings.npy', clue_context_embeddings),
]:
    mb = arr.nbytes / 1024**2
    print(f'{name:<35s} {str(arr.shape):<25s} {mb:>6.1f} MB')

total_mb = (definition_embeddings.nbytes + answer_embeddings.nbytes
            + clue_context_embeddings.nbytes) / 1024**2
print(f'\nTotal embedding memory: {total_mb:.1f} MB')
print(f'\nIndex sizes:')
print(f'  definition_index:     {len(definition_index):,} rows')
print(f'  answer_index:         {len(answer_index):,} rows')
print(f'  clue_context_index:   {len(clue_context_index):,} rows')

# --- Shape and consistency assertions ---
n_def = len(definition_index)
n_ans = len(answer_index)
n_cc = len(clue_context_index)

assert definition_embeddings.shape == (n_def, 3, EMBED_DIM), (
    f'definition_embeddings shape mismatch: expected ({n_def}, 3, {EMBED_DIM}), '
    f'got {definition_embeddings.shape}')
assert answer_embeddings.shape == (n_ans, 3, EMBED_DIM), (
    f'answer_embeddings shape mismatch: expected ({n_ans}, 3, {EMBED_DIM}), '
    f'got {answer_embeddings.shape}')
assert clue_context_embeddings.shape == (n_cc, EMBED_DIM), (
    f'clue_context_embeddings shape mismatch: expected ({n_cc}, {EMBED_DIM}), '
    f'got {clue_context_embeddings.shape}')

print(f'\nAll shape assertions passed.')

## Build Embedding Matrices

For each of the 240,211 rows in our working set, we need 7 embedding vectors
(each 1024-dim):

| Abbrev | Source | Slot in `.npy` array |
|--------|--------|---------------------|
| `w1_all` | definition allsense-average | `definition_embeddings[idx, 0, :]` |
| `w1_common` | definition common synset | `definition_embeddings[idx, 1, :]` |
| `w1_obscure` | definition obscure synset | `definition_embeddings[idx, 2, :]` |
| `w2_all` | answer allsense-average | `answer_embeddings[idx, 0, :]` |
| `w2_common` | answer common synset | `answer_embeddings[idx, 1, :]` |
| `w2_obscure` | answer obscure synset | `answer_embeddings[idx, 2, :]` |
| `w1_clue` | definition in clue context | `clue_context_embeddings[idx, :]` |

Rather than looping row-by-row, we build full `(N, 1024)` matrices for each
embedding type using bulk index lookups, then fancy-index into the `.npy`
arrays. For definition and answer embeddings, we map `definition_wn` /
`answer_wn` strings to their row positions in the index files. For
clue-context embeddings, we use the `cc_row_position` column recorded during
the merge — this is a direct positional index into the `.npy` array and
correctly handles double-definition clues (where multiple rows share the
same `clue_id` but have different clue-context embeddings).

In [ ]:
# --- Build word → row-position mappings for O(1) lookup ---
# definition_index and answer_index have integer row indices (0, 1, 2, ...)
# and a 'word' column. We create a Series mapping word string → row position.
def_word_to_idx = pd.Series(
    definition_index.index, index=definition_index['word'])
ans_word_to_idx = pd.Series(
    answer_index.index, index=answer_index['word'])

# --- Map each row's lookup key to its position in the embedding arrays ---
# .map() returns NaN for any key not found — we assert none are missing.
def_indices = df['definition_wn'].map(def_word_to_idx)
assert def_indices.notna().all(), (
    f'{def_indices.isna().sum()} definition_wn values not found in '
    f'definition_index. Examples: '
    f'{df.loc[def_indices.isna(), "definition_wn"].head().tolist()}')
def_indices = def_indices.astype(int).values

ans_indices = df['answer_wn'].map(ans_word_to_idx)
assert ans_indices.notna().all(), (
    f'{ans_indices.isna().sum()} answer_wn values not found in '
    f'answer_index. Examples: '
    f'{df.loc[ans_indices.isna(), "answer_wn"].head().tolist()}')
ans_indices = ans_indices.astype(int).values

# For clue-context embeddings, we use cc_row_position directly. This column
# was set to np.arange(len(cc_phrases)) before the merge, giving each row
# its position in clue_context_embeddings.npy. Unlike clue_context_index.csv
# (which only has clue_id), cc_row_position correctly handles double-definition
# clues where multiple rows share the same clue_id but have different
# clue-context embeddings.
cc_indices = df['cc_row_position'].values

# --- Fancy-index into embedding arrays to build (N, 1024) matrices ---
# Each matrix has one row per clue in our working set.
N = len(df)
print(f'Building 7 embedding matrices of shape ({N:,}, {EMBED_DIM})...')

w1_all     = definition_embeddings[def_indices, 0, :]   # (N, 1024)
w1_common  = definition_embeddings[def_indices, 1, :]   # (N, 1024)
w1_obscure = definition_embeddings[def_indices, 2, :]   # (N, 1024)
w2_all     = answer_embeddings[ans_indices, 0, :]       # (N, 1024)
w2_common  = answer_embeddings[ans_indices, 1, :]       # (N, 1024)
w2_obscure = answer_embeddings[ans_indices, 2, :]       # (N, 1024)
w1_clue    = clue_context_embeddings[cc_indices, :]     # (N, 1024)

print(f'Done. Each matrix: {w1_all.shape}')
print(f'Total memory for 7 matrices: '
      f'{7 * w1_all.nbytes / 1024**2:.1f} MB')

## Cosine Similarity Features (21 total)

We compute 21 pairwise cosine similarities organized into two groups:

**Context-Free Meaning (15):** All ${6 \choose 2} = 15$ pairwise cosine
similarities among the 6 context-free embeddings (3 definition × 3 answer
cross-word pairs = 9, plus 3 within-definition and 3 within-answer pairs).
These capture the semantic relationship between definition and answer
senses *without* any influence from the clue's surface reading.

**Context-Informed Meaning (6):** Cosine similarity between
`word1_clue_context` (the definition embedded within the clue surface
using CALE's `<t></t>` delimiters) and each of the 6 context-free
embeddings. These capture how the clue's surface reading shifts the
definition's embedding — the core signal for misdirection.

We use row-wise dot products on L2-normalized vectors rather than
`sklearn.metrics.pairwise.cosine_similarity` on individual pairs. This
computes all N similarities for a given pair of embedding types in one
vectorized operation, which is orders of magnitude faster than looping.

In [ ]:
def rowwise_cosine(A, B):
    """Compute row-wise cosine similarity between corresponding rows of A and B.

    Parameters
    ----------
    A, B : np.ndarray, shape (N, D)
        Two matrices with the same shape.

    Returns
    -------
    np.ndarray, shape (N,)
        Cosine similarity for each row pair: cos(A[i], B[i]).
    """
    # L2-normalize each row, then take the row-wise dot product.
    A_norm = np.linalg.norm(A, axis=1, keepdims=True)
    B_norm = np.linalg.norm(B, axis=1, keepdims=True)
    return np.sum((A / A_norm) * (B / B_norm), axis=1)


# --- Context-Free Meaning: 15 features ---
# Cross-word pairs (definition × answer): 3 × 3 = 9
df['cos_w1all_w2all']       = rowwise_cosine(w1_all, w2_all)
df['cos_w1all_w2common']    = rowwise_cosine(w1_all, w2_common)
df['cos_w1all_w2obscure']   = rowwise_cosine(w1_all, w2_obscure)
df['cos_w1common_w2all']    = rowwise_cosine(w1_common, w2_all)
df['cos_w1common_w2common'] = rowwise_cosine(w1_common, w2_common)
df['cos_w1common_w2obscure']= rowwise_cosine(w1_common, w2_obscure)
df['cos_w1obscure_w2all']   = rowwise_cosine(w1_obscure, w2_all)
df['cos_w1obscure_w2common']= rowwise_cosine(w1_obscure, w2_common)
df['cos_w1obscure_w2obscure'] = rowwise_cosine(w1_obscure, w2_obscure)

# Within-definition pairs: C(3,2) = 3
df['cos_w1all_w1common']    = rowwise_cosine(w1_all, w1_common)
df['cos_w1all_w1obscure']   = rowwise_cosine(w1_all, w1_obscure)
df['cos_w1common_w1obscure']= rowwise_cosine(w1_common, w1_obscure)

# Within-answer pairs: C(3,2) = 3
df['cos_w2all_w2common']    = rowwise_cosine(w2_all, w2_common)
df['cos_w2all_w2obscure']   = rowwise_cosine(w2_all, w2_obscure)
df['cos_w2common_w2obscure']= rowwise_cosine(w2_common, w2_obscure)

# --- Context-Informed Meaning: 6 features ---
# word1_clue_context paired with each of the 6 context-free embeddings.
df['cos_w1clue_w1all']      = rowwise_cosine(w1_clue, w1_all)
df['cos_w1clue_w1common']   = rowwise_cosine(w1_clue, w1_common)
df['cos_w1clue_w1obscure']  = rowwise_cosine(w1_clue, w1_obscure)
df['cos_w1clue_w2all']      = rowwise_cosine(w1_clue, w2_all)
df['cos_w1clue_w2common']   = rowwise_cosine(w1_clue, w2_common)
df['cos_w1clue_w2obscure']  = rowwise_cosine(w1_clue, w2_obscure)

print('Computed 21 cosine similarity features.')

## Verification: Cosine Similarity Features

In [ ]:
# --- Identify the 21 cosine columns ---
cos_cols = [c for c in df.columns if c.startswith('cos_')]

# Assert exactly 21 cosine similarity columns
assert len(cos_cols) == 21, (
    f'Expected 21 cosine similarity columns, found {len(cos_cols)}: {cos_cols}')

# Assert no NaN values in any cosine column (Decision 3: no NaN features).
# Cosine similarities are guaranteed non-NaN as long as embeddings are non-zero,
# which we verified in Step 2.
nan_counts = df[cos_cols].isnull().sum()
assert (nan_counts == 0).all(), (
    f'NaN values found in cosine columns:\n'
    f'{nan_counts[nan_counts > 0]}')

print(f'Verification passed: 21 cosine columns, 0 NaN values.')
print(f'Rows: {len(df):,}\n')

# --- Descriptive statistics ---
# Context-free features (15): cross-word similarities tend to be moderate
# (definition and answer are semantically related but not identical), while
# within-word similarities tend to be high (different senses of the same word
# still share much of the embedding space).
# Context-informed features (6): the clue context shifts the definition
# embedding — the degree of shift is the misdirection signal.
print('--- Descriptive Statistics (21 Cosine Similarity Features) ---\n')
stats = df[cos_cols].describe().T[['mean', 'std', 'min', 'max']]
# Use wider display so columns don't wrap
with pd.option_context('display.float_format', '{:.4f}'.format,
                       'display.max_colwidth', 30):
    print(stats.to_string())

print(f'\n--- First 5 Rows (Cosine Similarity Features) ---\n')
with pd.option_context('display.float_format', '{:.4f}'.format,
                       'display.max_columns', None):
    display(df[cos_cols].head())

## WordNet Relationship Features (22 total)

WordNet relationship features capture how the definition and answer words are
connected in the WordNet semantic hierarchy. Unlike cosine similarity features
(which measure embedding-space proximity), these features capture discrete
structural relationships — e.g., whether the answer is a hypernym (more
general category) or hyponym (more specific instance) of the definition.

Cryptic crossword setters often exploit these relationships: the definition
might be a hypernym of the answer (e.g., "flower" → ROSE), or the two might
share a two-hop path (e.g., "plant" and "aster" are both hyponyms of
"organism" — a co-hyponymy pattern captured by `hyponym_of_hypernym`). By
encoding *which* relationship paths exist, we give the classifier structural
information about the type of semantic connection — not just whether one exists.

**Feature breakdown:**
- **20 boolean features** (`wn_rel_*`): whether the answer is reachable from
  the definition via specific one-hop or two-hop WordNet relationships. Covers
  all relationship types from Table 3 of the design doc: synonym, hypernym,
  hyponym, meronyms, holonyms, and their two-hop compounds. The original plan
  estimated 19 types; we implement 20 to include `hypernym_of_hyponym` which
  also appears in the data.
- **`wn_max_path_sim`** (float): maximum path similarity across all
  definition–answer synset pairs. Uses `path_similarity()` for consistency
  with Hans's preliminary findings, where `wn_path_sim` was the single most
  predictive feature (removing it alone dropped accuracy by 3.4%). Default 0.0
  if no synsets connect.
- **`wn_shared_synset_count`** (int): number of synsets shared by both
  definition and answer words (both appear as lemmas in the same synset).
  Default 0.

**Missing value handling (Decision 3):** Pairs with no WordNet connection get
False for all 20 booleans, 0.0 for path similarity, 0 for shared synset count.
No NaN values are ever produced.

**Optimization:** Many rows share the same (definition_wn, answer_wn) pair —
different clues for the same definition–answer combination. Since relationship
features depend only on the word pair (not the clue text), we compute them once
per unique pair and merge back, avoiding redundant WordNet traversals.

**Implementation note (Decision 18):** All functions are standalone so they can
later be extracted into `scripts/feature_utils.py` for reuse when computing
features for distractor pairs in Steps 5 and 7.

In [ ]:
# ============================================================
# WordNet Relationship Feature Functions
# ============================================================
# These functions are standalone and can be extracted into
# scripts/feature_utils.py for reuse in distractor feature
# computation (Steps 5 and 7). See Decision 18.
# ============================================================


def get_wordnet_synsets(word):
    """Look up all WordNet synsets for a word, handling multi-word entries.

    Tries the word as-is first (which works for both single-word and
    underscore-separated multi-word entries like "ice_cream"). If no
    synsets are found and the word contains spaces, retries with spaces
    converted to underscores — WordNet's convention for multi-word entries.

    Parameters
    ----------
    word : str
        The word to look up (from the definition_wn or answer_wn column).

    Returns
    -------
    list of nltk.corpus.reader.wordnet.Synset
        All synsets found for the word. Empty list if no synsets exist.
    """
    synsets = wn.synsets(word)
    if not synsets and ' ' in word:
        synsets = wn.synsets(word.replace(' ', '_'))
    return synsets


def check_synonym(def_synsets, ans_word):
    """Check if the answer word is a synonym of the definition in WordNet.

    Two words are WordNet synonyms if they share at least one synset — i.e.,
    the answer word appears as a lemma in some synset of the definition word.
    This is a lemma-level check, not a synset-graph traversal: we look at
    the set of lemma names within each definition synset and check whether
    the answer word is among them.

    Parameters
    ----------
    def_synsets : list of Synset
        All synsets for the definition word.
    ans_word : str
        The answer word (lowercase, underscored for multi-word; from the
        answer_wn column).

    Returns
    -------
    bool
        True if the answer appears as a lemma in any definition synset.
    """
    for syn in def_synsets:
        lemma_names = {lemma.name().lower() for lemma in syn.lemmas()}
        if ans_word in lemma_names:
            return True
    return False


def check_synset_reachable(def_synsets, ans_synsets_set, hops):
    """Check if any answer synset is reachable from definition synsets via
    a sequence of WordNet relationship hops.

    For single-hop relationships (e.g., "hyponym"), ``hops`` contains one
    method name — we follow that relationship from each definition synset and
    check if any answer synset appears among the targets. For compound
    two-hop relationships (e.g., "hyponym_of_hypernym"), ``hops`` contains
    two method names: the first hop follows the second word of the compound
    name (hypernyms), then the second hop follows the first word (hyponyms).
    This right-to-left reading matches the English semantics: "hyponym OF
    hypernym" = take hypernyms first, then take hyponyms of those.

    Parameters
    ----------
    def_synsets : list of Synset
        Starting synsets (from the definition word).
    ans_synsets_set : set of Synset
        Target synsets (from the answer word).
    hops : list of str
        Sequence of WordNet synset method names to follow. Each must be
        a valid method on nltk.corpus.reader.wordnet.Synset (e.g.,
        'hypernyms', 'hyponyms', 'part_meronyms').

    Returns
    -------
    bool
        True if any answer synset is reachable via the specified path.
    """
    current = set(def_synsets)
    for method_name in hops:
        next_level = set()
        for synset in current:
            next_level.update(getattr(synset, method_name)())
        current = next_level
        if not current:
            return False
    return bool(current & ans_synsets_set)


def compute_max_path_similarity(def_synsets, ans_synsets):
    """Compute the maximum path similarity across all definition-answer
    synset pairs.

    Path similarity (Rada et al., 1989) measures the inverse of the shortest
    path length between two synsets in the WordNet hypernym/hyponym hierarchy,
    normalized to [0, 1]. We use path_similarity (not wup_similarity) for
    consistency with Hans's preliminary classification experiments, where
    wn_path_sim was the single most predictive feature.

    Parameters
    ----------
    def_synsets : list of Synset
        All synsets for the definition word.
    ans_synsets : list of Synset
        All synsets for the answer word.

    Returns
    -------
    float
        Maximum path similarity across all synset pairs. Returns 0.0 if
        no synset pair connects or if either word has no synsets.
    """
    max_sim = 0.0
    for ds in def_synsets:
        for as_ in ans_synsets:
            sim = ds.path_similarity(as_)
            if sim is not None and sim > max_sim:
                max_sim = sim
    return max_sim


def compute_shared_synset_count(def_synsets, ans_synsets):
    """Count synsets that contain both the definition and answer as lemmas.

    A shared synset means both words can express the same concept — they are
    direct synonyms for that particular meaning. This is related to but
    distinct from the synonym boolean: check_synonym looks at lemma names
    within synsets, while this counts the actual synset overlap. In practice
    these are equivalent, but shared_synset_count captures *how many* senses
    overlap, not just whether any do.

    Parameters
    ----------
    def_synsets : list of Synset
        All synsets for the definition word.
    ans_synsets : list of Synset
        All synsets for the answer word.

    Returns
    -------
    int
        Number of synsets appearing in both lists. Returns 0 if no overlap.
    """
    return len(set(def_synsets) & set(ans_synsets))


# --- Relationship type definitions ---
# Maps each relationship type name to the sequence of WordNet synset methods
# needed to check reachability. For compound types "X_of_Y", the first hop
# follows Y and the second hop follows X (read right-to-left). For example,
# "hyponym_of_hypernym" means: from definition synsets, follow hypernyms()
# (first hop), then follow hyponyms() of those results (second hop), and
# check if any answer synset appears in the final set.
#
# These 19 types (plus synonym handled separately = 20 total) cover all
# relationship types from Table 3 of the design doc.
RELATIONSHIP_HOPS = {
    # --- Single-hop relationships (6) ---
    # Direct taxonomic and part-whole relationships.
    'hyponym':           ['hyponyms'],
    'hypernym':          ['hypernyms'],
    'part_holonym':      ['part_holonyms'],
    'part_meronym':      ['part_meronyms'],
    'substance_meronym': ['substance_meronyms'],
    'member_meronym':    ['member_meronyms'],

    # --- Two-hop relationships (13) ---
    # Taxonomic two-hop: moving up/down the is-a hierarchy twice.
    'hyponym_of_hypernym':  ['hypernyms', 'hyponyms'],   # co-hyponymy
    'hypernym_of_hyponym':  ['hyponyms', 'hypernyms'],   # co-hypernymy
    'hyponym_of_hyponym':   ['hyponyms', 'hyponyms'],    # grandchild
    'hypernym_of_hypernym': ['hypernyms', 'hypernyms'],  # grandparent

    # Mixed taxonomic + part-whole: one taxonomic hop + one part-whole hop.
    'part_holonym_of_hyponym':      ['hyponyms', 'part_holonyms'],
    'hyponym_of_part_holonym':      ['part_holonyms', 'hyponyms'],
    'substance_meronym_of_hyponym': ['hyponyms', 'substance_meronyms'],
    'part_meronym_of_hyponym':      ['hyponyms', 'part_meronyms'],
    'hyponym_of_part_meronym':      ['part_meronyms', 'hyponyms'],
    'part_meronym_of_hypernym':     ['hypernyms', 'part_meronyms'],
    'part_holonym_of_hypernym':     ['hypernyms', 'part_holonyms'],

    # Part-whole two-hop: two part-whole hops.
    'part_holonym_of_part_meronym':     ['part_meronyms', 'part_holonyms'],
    'member_meronym_of_member_holonym': ['member_holonyms', 'member_meronyms'],
}


def compute_wordnet_relationship_features(def_word, ans_word):
    """Compute all WordNet relationship features for a (definition, answer) pair.

    This is the master function that combines all relationship checks into
    a single feature dictionary. It is designed to be called once per unique
    (definition_wn, answer_wn) pair; the results are then merged back to all
    rows sharing that pair.

    Parameters
    ----------
    def_word : str
        The definition word (lowercase, underscored for multi-word; from the
        definition_wn column).
    ans_word : str
        The answer word (lowercase, underscored for multi-word; from the
        answer_wn column).

    Returns
    -------
    dict
        Keys are feature column names, values are feature values:
        - 20 boolean features (int 0/1): wn_rel_synonym, wn_rel_hyponym, ...
        - wn_max_path_sim (float): max path similarity, default 0.0
        - wn_shared_synset_count (int): shared synset count, default 0
    """
    def_synsets = get_wordnet_synsets(def_word)
    ans_synsets = get_wordnet_synsets(ans_word)
    ans_synsets_set = set(ans_synsets)

    features = {}

    # --- Synonym (lemma-based, one hop) ---
    features['wn_rel_synonym'] = int(check_synonym(def_synsets, ans_word))

    # --- Synset-based relationship checks (one-hop and two-hop) ---
    for rel_name, hops in RELATIONSHIP_HOPS.items():
        features[f'wn_rel_{rel_name}'] = int(
            check_synset_reachable(def_synsets, ans_synsets_set, hops)
        )

    # --- Max path similarity ---
    features['wn_max_path_sim'] = compute_max_path_similarity(
        def_synsets, ans_synsets)

    # --- Shared synset count ---
    features['wn_shared_synset_count'] = compute_shared_synset_count(
        def_synsets, ans_synsets)

    return features


# Quick sanity check: verify the function produces the expected keys
# and sensible values for a known word pair.
_test = compute_wordnet_relationship_features('flower', 'rose')
_expected_keys = (
    ['wn_rel_synonym']
    + [f'wn_rel_{r}' for r in RELATIONSHIP_HOPS]
    + ['wn_max_path_sim', 'wn_shared_synset_count']
)
assert set(_test.keys()) == set(_expected_keys), (
    f'Key mismatch. Expected {len(_expected_keys)} keys, got {len(_test)}')
print(f'Function sanity check passed (flower → rose):')
print(f'  wn_rel_synonym={_test["wn_rel_synonym"]}, '
      f'wn_rel_hypernym={_test["wn_rel_hypernym"]}, '
      f'wn_rel_hyponym={_test["wn_rel_hyponym"]}, '
      f'wn_max_path_sim={_test["wn_max_path_sim"]:.4f}, '
      f'wn_shared_synset_count={_test["wn_shared_synset_count"]}')
print(f'  Total features per pair: {len(_test)}')

In [ ]:
# --- Deduplicate to unique (definition_wn, answer_wn) pairs ---
# Many clue rows share the same definition–answer pair (different clue
# sentences for the same word pair). Relationship features depend only on
# the word pair, not the clue text, so we compute them once per unique pair
# and merge back. This avoids redundant WordNet traversals and dramatically
# reduces computation time.

unique_pairs = df[['definition_wn', 'answer_wn']].drop_duplicates()
print(f'Total rows:                              {len(df):>10,}')
print(f'Unique (definition_wn, answer_wn) pairs: {len(unique_pairs):>10,}')
print(f'Deduplication ratio:                     {len(df) / len(unique_pairs):>10.2f}x')
print(f'\nComputing WordNet relationship features for {len(unique_pairs):,} '
      f'unique pairs...')
print(f'(This may take 10–30 minutes on CPU.)\n')

# --- Compute features for each unique pair ---
# Use itertuples (faster than iterrows) to iterate over the unique pairs.
pair_list = list(unique_pairs.itertuples(index=False, name=None))
results = []
for def_word, ans_word in tqdm(pair_list, desc='WordNet relationships'):
    feats = compute_wordnet_relationship_features(def_word, ans_word)
    feats['definition_wn'] = def_word
    feats['answer_wn'] = ans_word
    results.append(feats)

rel_features_df = pd.DataFrame(results)
print(f'\nComputed {len(rel_features_df):,} unique-pair feature rows.')

# --- Merge relationship features back to all rows ---
# Left merge preserves all 240,211 rows in df. Each row gets the
# relationship features for its (definition_wn, answer_wn) pair.
n_before = len(df)
df = df.merge(rel_features_df, on=['definition_wn', 'answer_wn'], how='left')
assert len(df) == n_before, (
    f'Merge changed row count: {n_before:,} → {len(df):,}. '
    f'This should not happen with a left merge on unique keys.')

print(f'Merged relationship features back to {len(df):,} rows.')

## Verification: WordNet Relationship Features

In [ ]:
# --- Identify relationship columns ---
wn_rel_bool_cols = sorted([c for c in df.columns if c.startswith('wn_rel_')])
wn_continuous_cols = ['wn_max_path_sim', 'wn_shared_synset_count']
wn_all_cols = wn_rel_bool_cols + wn_continuous_cols

print(f'Boolean relationship columns ({len(wn_rel_bool_cols)}):')
for col in wn_rel_bool_cols:
    print(f'  {col}')
print(f'\nContinuous relationship columns ({len(wn_continuous_cols)}):')
for col in wn_continuous_cols:
    print(f'  {col}')
print(f'\nTotal relationship columns: {len(wn_all_cols)}')

# --- Assert expected counts ---
# The design doc Table 3 lists 20 relationship types (19 from the original
# plan + hypernym_of_hyponym). Together with path similarity and shared
# synset count, that gives 22 total relationship columns — one more than
# the original PLAN.md estimate of 21 (which assumed 19 boolean types).
assert len(wn_rel_bool_cols) == 20, (
    f'Expected 20 boolean relationship columns, found {len(wn_rel_bool_cols)}')
assert len(wn_all_cols) == 22, (
    f'Expected 22 total relationship columns, found {len(wn_all_cols)}')

# --- Assert no NaN values (Decision 3) ---
nan_counts = df[wn_all_cols].isnull().sum()
assert (nan_counts == 0).all(), (
    f'NaN values found in relationship columns:\n'
    f'{nan_counts[nan_counts > 0]}')
print(f'\nVerification passed: {len(wn_rel_bool_cols)} boolean + '
      f'{len(wn_continuous_cols)} continuous = {len(wn_all_cols)} '
      f'relationship columns, 0 NaN values.')
print(f'Rows: {len(df):,}\n')

# --- Boolean feature prevalence ---
# Which relationship types are most common? We expect hyponym and synonym
# to dominate, since cryptic crossword definitions typically use "is-a"
# relationships (hypernym→answer) or direct synonymy.
print(f'--- Boolean Feature Prevalence ---')
print(f'{"Feature":<45s} {"True":>8s} {"False":>8s} {"% True":>8s}')
print(f'{"-"*45} {"-"*8} {"-"*8} {"-"*8}')
for col in wn_rel_bool_cols:
    n_true = int(df[col].sum())
    n_false = len(df) - n_true
    pct = 100 * n_true / len(df)
    print(f'{col:<45s} {n_true:>8,} {n_false:>8,} {pct:>7.2f}%')

# --- Rows with at least one boolean True ---
any_true = df[wn_rel_bool_cols].any(axis=1).sum()
pct_any = 100 * any_true / len(df)
print(f'\nRows with at least one boolean True: '
      f'{any_true:,} / {len(df):,} ({pct_any:.1f}%)')
print(f'(Design doc preliminary finding: ~31–34%)')

# --- Continuous feature statistics ---
print(f'\n--- Continuous Feature Statistics ---\n')
with pd.option_context('display.float_format', '{:.4f}'.format):
    print(df[wn_continuous_cols].describe().to_string())

# --- Distribution of wn_shared_synset_count ---
print(f'\n--- wn_shared_synset_count value distribution ---')
ssc_counts = df['wn_shared_synset_count'].value_counts().sort_index()
for val, cnt in ssc_counts.items():
    print(f'  {val}: {cnt:,} rows ({100 * cnt / len(df):.2f}%)')

### Interpretation

**WordNet connectivity:** The percentage of rows with at least one boolean
relationship True should be compared to the ~31–34% reported in Hans's
preliminary analysis (design doc Table 3). Any difference likely reflects the
broader dataset (we include multi-word entries and double-definition clues
that the preliminary analysis excluded) and the improved WordNet lookup
(underscore conversion for multi-word entries — see Decision 17).

**Relationship type distribution:** Cryptic crossword definitions typically use
hypernym relationships ("flower" for ROSE, "creature" for CAT) or direct
synonymy ("hide" for CONCEAL). `hyponym_of_hypernym` (co-hyponymy) captures
pairs like "plant"/"aster" — both children of a shared parent concept — which
is common when the definition is a sibling category rather than a direct
parent. Two-hop types involving meronyms and holonyms are rarer since
part–whole relationships are less common in crossword definitions.

**Path similarity:** `wn_max_path_sim` was the single most predictive feature
in Hans's preliminary experiments (removing it alone dropped accuracy by 3.4%).
Its distribution — including the proportion of zero values where no WordNet
path exists — informs whether this feature will remain dominant with the
expanded 46-feature set. Values near 1.0 indicate direct synonym or
parent–child relationships; values near 0.05–0.1 indicate distant connections.

**Shared synsets:** Pairs with `wn_shared_synset_count > 0` are definitional
synonyms — the definition and answer literally share a WordNet sense — making
classification trivially easy for those rows. The proportion of such pairs
contributes to the baseline difficulty of the classification task.